In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import math
from numpy import genfromtxt
import numpy as np
from uncertainties import unumpy as unp
import matplotlib.pyplot as plt
from scipy.interpolate import RectBivariateSpline, make_interp_spline 

from pisa import FTYPE, ureg
from pisa.utils.fileio import from_file

In [ ]:
params = {'figure.figsize': (7, 7*0.618),
          'legend.fontsize': 14,
          'axes.labelsize': 16,
          'axes.titlesize': 16,
          'xtick.labelsize': 16,
          'ytick.labelsize': 16}
plt.rcParams.update(params)

def get_range(x, y, cut):
    xs = x[np.where(y < cut)]
    return max(xs) - min(xs)

### numu disappearance

In [ ]:
sin_sqs = np.linspace(0.4, 0.6, 11)
delta_ms = np.linspace(2.35e-3, 2.6e-3, 11)

idxs = np.reshape(np.stack(np.meshgrid(sin_sqs, delta_ms)).T, (121,2)).astype(np.float32)

chi2, chi2_det1, chi2_det2 = np.zeros((11,11)), np.zeros((11,11)), np.zeros((11,11))
for i, idx in enumerate(idxs):
    res = from_file('../scans/numu_disappearance/3_years/upgrade_fit_deltam31_%s_theta23_%s.json'%(idx[1], idx[0]))
    chi2[i%11][int(i/11)] = res['metric_val']
    chi2_det1[i%11][int(i/11)] = res['detailed_metric_info'][0]['mod_chi2']['maps']['total']
    chi2_det2[i%11][int(i/11)] = res['detailed_metric_info'][1]['mod_chi2']['maps']['total']
'''
chi2_DC = np.zeros((11,11))
for i, idx in enumerate(idxs):
    res = from_file('../scans/numu_disappearance/DC_only/12_years/dc_fit_deltam31_%s_theta23_%s_.json'%(idx[1], idx[0]))
    chi2_DC[i%11][int(i/11)] = res['metric_val']
'''
for p in res['params']:
    if p['name'] == 'deltam31':
        deltam31_true = p['nominal_value'].magnitude
    elif p['name'] == 'theta23':
        theta23_true = p['nominal_value'].magnitude

In [ ]:
interp = RectBivariateSpline(sin_sqs, delta_ms, chi2)

interp_xs = np.linspace(sin_sqs[0], sin_sqs[-1], 100)
interp_ys = np.linspace(delta_ms[0], delta_ms[-1], 100)
interp_zs = interp(interp_xs, interp_ys)

interp_det1 = RectBivariateSpline(sin_sqs, delta_ms, chi2_det1)
interp_det2 = RectBivariateSpline(sin_sqs, delta_ms, chi2_det2)

interp_zs_det1 = interp_det1(interp_xs, interp_ys)
interp_zs_det2 = interp_det2(interp_xs, interp_ys)
    
#interp_DC = RectBivariateSpline(sin_sqs, delta_ms, chi2_DC)
#interp_zs_DC = interp_DC(interp_xs, interp_ys)

In [ ]:
level, lab = 4.61, '90%'

C = plt.contour(interp_xs, interp_ys, interp_zs_det2, [level], linestyles=['--'], colors=['Tab:orange'])
plt.clabel(C, C.levels, inline=True, fmt={level: lab}, fontsize=10)
plt.plot([], [], label='DeepCore (12yr)', linestyle='--', color='Tab:orange')
    
C = plt.contour(interp_xs, interp_ys, interp_zs_det1, [level], linestyles=['--'], colors=['Tab:blue'])
plt.clabel(C, C.levels, inline=True, fmt={level: lab}, fontsize=10)
plt.plot([], [], label='DeepCore +\nnew strings (3yr)', linestyle='--', color='Tab:blue')

#C = plt.contour(interp_xs, interp_ys, interp_zs_DC, [level], colors=['black'], linewidths=[2], linestyles=[':'])
#plt.clabel(C, C.levels, inline=True, fmt={level: lab}, fontsize=10)
#plt.plot([], [], label='2028 w/o new strings', color='black', linewidth=2, linestyle=':')

C = plt.contour(interp_xs, interp_ys, interp_zs, [level], colors=['black'], linewidths=[2])
plt.clabel(C, C.levels, inline=True, fmt={level: lab}, fontsize=10)
plt.plot([], [], label='2028 w/ new strings', color='black', linewidth=2)

plt.scatter(np.sin(theta23_true * math.pi/180)**2, 
            deltam31_true, 
            color='r', marker='x', label='Injected truth')

#plt.title('No detector and ice systematics')
plt.legend(prop={'size':13}, bbox_to_anchor = [1.3, 1.0])
plt.xlabel(r'$\sin^2(\theta_{23})$')
plt.xlim(0.42, 0.6)
#plt.xticks([])
plt.ylabel('$\Delta m_{31}^2$ ($e\mathrm{V}^2$)', fontsize=16)
plt.ylim(2.4e-3, 2.57e-3)
#plt.yticks([])
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

#plt.savefig('../plots/scan_dynedge_all.png', bbox_inches='tight')

In [ ]:
plt.figure(figsize=(2*7, 7*0.618))

plt.subplot(121)
plt.plot(interp_xs, np.min(interp_zs, axis=0), color='black', linewidth=2, label='2028 w/ new strings')
plt.plot(interp_xs, np.min(interp_zs_DC, axis=0), color='black', linewidth=2, linestyle=':', label='2028 w/o new strings')
plt.axvline(np.sin(theta23_true * math.pi/180)**2, color='r', label='Injected truth')
plt.axhline(9, color='grey', linestyle=':')
plt.text(np.sin(theta23_true * math.pi/180)**2, 10, '~15% improved', size=16, ha='center')
plt.title('No HS or new systematics')
plt.xlabel(r'$\sin^2(\theta_{23})$')
plt.ylabel('$\chi^{2}_{mod}$', fontsize=16)
plt.ylim(0,65)
plt.legend()

plt.subplot(122)
plt.plot(interp_ys, np.min(interp_zs, axis=1), color='black', linewidth=2, label='2028 w/ new strings')
plt.plot(interp_ys, np.min(interp_zs_DC, axis=1), color='black', linewidth=2, linestyle=':', label='2028 w/o new strings')
plt.axvline(deltam31_true, color='r', label='Injected truth')
plt.axhline(9, color='grey', linestyle=':')
plt.text(deltam31_true, 10, '~25% improved', size=16, ha='center')
plt.title('No HS or new systematics')
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
plt.xlabel('$\Delta m_{31}^2$ ($e\mathrm{V}^2$)', fontsize=16)
plt.ylabel('$\chi^{2}_{mod}$', fontsize=16)
plt.ylim(0,65)

#zplt.savefig('../plots/scan_dynedge_1D.png', bbox_inches='tight')

In [ ]:
np.round(1 - get_range(interp_xs, np.min(interp_zs, axis=0), 9) / get_range(interp_xs, np.min(interp_zs_DC, axis=0), 9), 2)

In [ ]:
np.round(1 - get_range(interp_ys, np.min(interp_zs, axis=1), 9) / get_range(interp_ys, np.min(interp_zs_DC, axis=1), 9), 2)

old sensitivities

In [ ]:
old_sens = genfromtxt('../scans/numu_diss_old.csv', delimiter=',')

In [ ]:
plt.plot(old_sens[:,0]-0.51, old_sens[:,1]-0.0023, linewidth=3, color='r', label='old')

C = plt.contour(interp_xs-np.sin(theta23_true * math.pi/180)**2, interp_ys-deltam31_true, interp_zs_det1, [4.61], linestyles=['--'], colors=['Tab:blue'])
plt.clabel(C, C.levels, inline=True, fmt={4.61: '90%'}, fontsize=10)
plt.plot([], [], label='new', linestyle='--', color='Tab:blue')

plt.scatter(0, 0, color='r', marker='x', label='Injected truth')

plt.legend()
plt.xticks([])
plt.yticks([])
plt.xlabel(r'$\sin^2(\theta_{23})$')
plt.ylabel('$\Delta m_{31}^2$ ($e\mathrm{V}^2$)')

#plt.savefig('../plots/scan_dynedge_old_new.png', bbox_inches='tight')

### nutau_appearance

In [ ]:
tau_norms = np.linspace(0.85, 1.15, 11)

chi2, chi2_det1, chi2_det2 = [], [], []
for f in np.sort(os.listdir('../scans/nutau_appearance/3_years/'))[1:]:
    res = from_file('../scans/nutau_appearance/3_years/'+f)
    chi2.append(res['metric_val'])
    chi2_det1.append(res['detailed_metric_info'][0]['mod_chi2']['maps']['total'])
    chi2_det2.append(res['detailed_metric_info'][1]['mod_chi2']['maps']['total'])
    
#chi2_DC = []
#for f in np.sort(os.listdir('../scans/nutau_appearance/DC_only/12_years/'))[1:]:
#    res = from_file('../scans/nutau_appearance/DC_only/12_years/'+f)
#    chi2_DC.append(res['metric_val'])

In [ ]:
interp = make_interp_spline(tau_norms, chi2, k=3)

interp_xs = np.linspace(tau_norms[0], tau_norms[-1], 100)
interp_ys = interp(interp_xs)

interp_det1 = make_interp_spline(tau_norms, chi2_det1, k=3)
interp_det2 = make_interp_spline(tau_norms, chi2_det2, k=3)

interp_ys_det1 = interp_det1(interp_xs)
interp_ys_det2 = interp_det2(interp_xs)
    
#interp_DC = make_interp_spline(tau_norms, chi2_DC, k=3)
#interp_ys_DC = interp_DC(interp_xs)

In [ ]:
plt.plot(interp_xs, interp_ys_det2, label='DeepCore (12yr)', linestyle='--', color='Tab:orange')
plt.plot(interp_xs, interp_ys_det1, label='DeepCore + new strings (3yr)', linestyle='--', color='Tab:blue')
    
plt.plot(interp_xs, interp_ys, label='2028 w/ new strings', color='black', linewidth=2)

#plt.plot(interp_xs, interp_ys_DC, label='2028 w/o new strings', color='black', linewidth=2, linestyle=':')

plt.axvline(1, color='r', label='Injected truth')
plt.axhline(9, color='grey', linestyle=':')
#plt.text(1, 10, '~30% improved', size=16, ha='center')

plt.xlabel(r'$\nu_{\tau}$ norm')
plt.xlim(0.85,1.15)
plt.ylabel('$\chi^{2}_{mod}$', fontsize=16)
#plt.ylim(0,30)
#plt.title('No HS or new systematics')
plt.legend(prop={'size':14})

#plt.savefig('../plots/scan_tau_dynedge_all.png', bbox_inches='tight')

In [ ]:
np.round(1 - get_range(interp_xs, interp_ys, 9) / get_range(interp_xs, interp_ys_DC, 9), 2)

### NMO

In [ ]:
def calc_sens(TO, WO=None):
    if WO is None:
        return np.sqrt(TO)
    else:
        return np.sqrt(2)*(TO+WO)/np.sqrt(8*WO)

In [ ]:
best_fit_info_nh = from_file('../scans/NMO/upgrade_NMO_nh_nh_3.json')
best_fit_info_ih = from_file('../scans/NMO/upgrade_NMO_ih_nh_3.json')

best_fit_info_nh_DC_12 = from_file('../scans/NMO/dc_NMO_nh_nh_12.json')
best_fit_info_ih_DC_12 = from_file('../scans/NMO/dc_NMO_ih_nh_12.json')
best_fit_info_nh_DC_15 = from_file('../scans/NMO/dc_NMO_nh_nh_15.json')
best_fit_info_ih_DC_15 = from_file('../scans/NMO/dc_NMO_ih_nh_15.json')

In [ ]:
nh_asi_ICU = best_fit_info_nh['detailed_metric_info'][0]['mod_chi2']['maps']['total']
nh_asi_DC = best_fit_info_nh['detailed_metric_info'][1]['mod_chi2']['maps']['total']
nh_prior = sum(best_fit_info_nh['detailed_metric_info'][0]['mod_chi2']['priors'])
nh_asi = best_fit_info_nh['metric_val']
nh_asi_DC_only_12 = best_fit_info_nh_DC_12['metric_val']
nh_asi_DC_only_15 = best_fit_info_nh_DC_15['metric_val']

ih_asi_ICU = best_fit_info_ih['detailed_metric_info'][0]['mod_chi2']['maps']['total']
ih_asi_DC = best_fit_info_ih['detailed_metric_info'][1]['mod_chi2']['maps']['total']
ih_prior = sum(best_fit_info_ih['detailed_metric_info'][0]['mod_chi2']['priors'])
ih_asi = best_fit_info_ih['metric_val']
ih_asi_DC_only_12 = best_fit_info_ih_DC_12['metric_val']
ih_asi_DC_only_15 = best_fit_info_ih_DC_15['metric_val']

In [ ]:
sens_ICU = calc_sens(nh_asi_ICU+nh_prior, ih_asi_ICU+ih_prior)
sens_DC = calc_sens(nh_asi_DC+nh_prior, ih_asi_DC+ih_prior)

sens = calc_sens(nh_asi, ih_asi)
sens_DC_only_12 = calc_sens(nh_asi_DC_only_12, ih_asi_DC_only_12)
sens_DC_only_15 = calc_sens(nh_asi_DC_only_15, ih_asi_DC_only_15)

In [ ]:
plt.figure(figsize=(9, 5*0.618))

plt.scatter(range(5), [sens_DC_only_12, sens_DC, sens_ICU, sens, sens_DC_only_15])
for i, s in enumerate([sens_DC_only_12, sens_DC, sens_ICU, sens, sens_DC_only_15]):
    plt.text(i, s+0.1, '%.2f'%(s), ha='center', size=14)
#plt.axvline(2.5, color='black', linestyle='--')

plt.title('NO = True, no HS or new systematics')
plt.xlim(-0.5, 4.5)
plt.xticks(range(5), ['2025', 'DeepCore \n(12yr)', 'DeepCore + \nnew strings (3yr)', '2028 w/ \nnew strings', '2028 w/o \nnew strings'], size=13)
plt.ylim(0, 3.5)
plt.ylabel(r'$\sigma_{NMO}$')

#plt.savefig('../plots/NMO_dynedge.png', bbox_inches='tight')

livetime

In [ ]:
def get_sens(years, DC=False):
    if DC:
        best_fit_info_nh = from_file('../scans/NMO/dc_NMO_nh_nh_'+str(int(years))+'.json')
        best_fit_info_ih = from_file('../scans/NMO/dc_NMO_ih_nh_'+str(int(years))+'.json')
        
        nh_asi = best_fit_info_nh['metric_val']
        ih_asi = best_fit_info_ih['metric_val']
        
        return calc_sens(nh_asi, ih_asi)
    else:
        best_fit_info_nh = from_file('../scans/NMO/upgrade_NMO_nh_nh_'+str(int(years))+'.json')
        best_fit_info_ih = from_file('../scans/NMO/upgrade_NMO_ih_nh_'+str(int(years))+'.json')

        nh_asi_ICU = best_fit_info_nh['detailed_metric_info'][0]['mod_chi2']['maps']['total']
        nh_asi_DC = best_fit_info_nh['detailed_metric_info'][1]['mod_chi2']['maps']['total']
        nh_prior = sum(best_fit_info_nh['detailed_metric_info'][0]['mod_chi2']['priors'])
        nh_asi = best_fit_info_nh['metric_val']

        ih_asi_ICU = best_fit_info_ih['detailed_metric_info'][0]['mod_chi2']['maps']['total']
        ih_asi_DC = best_fit_info_ih['detailed_metric_info'][1]['mod_chi2']['maps']['total']
        ih_prior = sum(best_fit_info_ih['detailed_metric_info'][0]['mod_chi2']['priors'])
        ih_asi = best_fit_info_ih['metric_val']

        sens_ICU = calc_sens(nh_asi_ICU+nh_prior, ih_asi_ICU+ih_prior)
        sens_DC = calc_sens(nh_asi_DC+nh_prior, ih_asi_DC+ih_prior)
        sens = calc_sens(nh_asi, ih_asi)

        return sens, sens_ICU, sens_DC

In [ ]:
years_ICU, years_DC = np.arange(1,6), np.arange(1,18)

sens, sens_ICU, sens_DC = [], [], []
for i in years_ICU:
    a, b, c = get_sens(i)
    sens.append(a)
    sens_ICU.append(b)
    sens_DC.append(c)
    
sens_DC_only = []
for i in years_DC:
    sens_DC_only.append(get_sens(i, DC=True))

In [ ]:
plt.plot(years_ICU+12, sens, label='combined')
plt.plot(years_ICU+12, sens_ICU, label='Upgrade')
plt.plot(years_ICU+12, sens_DC, label='DeepCore')
plt.plot(years_DC, sens_DC_only, label='DeepCore only')

plt.axhline(1, linestyle='--', color='grey')
plt.axhline(3, linestyle='--', color='grey')
plt.axhline(5, linestyle='--', color='grey')
plt.axvline(12, linestyle=':', color='grey', label='New string deployment')
plt.legend()

plt.title('NO=True')
plt.xlim(1, 17)
plt.xlabel('years')
plt.ylim(0, 6)
plt.ylabel(r'NMO sensitivity ($\sigma$)')

#plt.savefig('../plots/NMO_dynedge_livetime_NO.png', bbox_inches='tight')